<a href="https://colab.research.google.com/github/somerstep/CARROT/blob/main/LLMJudge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
from datasets import load_dataset
import pandas as pd
import numpy as np
import ast

map = {'Idavidrein/gpqa/gpqa_extended':'gpqa', 'TAUR-Lab/MuSR':'musr',
       'TAUR-Lab/MuSR/object_placements':'musr', 'TAUR-Lab/MuSR/team_allocation':'musr',
       'TIGER-Lab/MMLU-Pro':'mmlu', 'lighteval/MATH/all':'math',
       'lighteval/MATH/all/test':'math', 'openhermes/teknium':'openhermes/teknium',
       'rungalileo/ragbench/covidqa':'ragbench', 'rungalileo/ragbench/cuad':'ragbench',
       'rungalileo/ragbench/delucionqa':'ragbench', 'rungalileo/ragbench/emanual':'ragbench',
       'rungalileo/ragbench/expertqa':'ragbench', 'rungalileo/ragbench/finqa':'ragbench',
       'rungalileo/ragbench/hagrid':'ragbench', 'rungalileo/ragbench/hotpotqa':'ragbench',
       'rungalileo/ragbench/msmarco':'ragbench', 'rungalileo/ragbench/pubmedqa':'ragbench',
       'rungalileo/ragbench/tatqa':'ragbench', 'rungalileo/ragbench/techqa':'ragbench'}

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("CARROT-LLM-Routing/SPROUT")
ds = ds['test'].to_pandas()
ds['dataset'] = [map[x] for x in ds.dataset]

In [10]:
n_models = 13
k = 6

In [11]:
np.random.seed(0)
cols = [int(np.random.choice(range(k,k+n_models))) for _ in range(ds.shape[0])]
resps = [ds.iloc[i,cols[i]] for i in range(ds.shape[0])]
ds = ds.iloc[:,:k]
ds['resps'] = resps

ds['llm_score'] = [x['score'] for x in ds.resps]
ds = ds.loc[ds.dataset!='openhermes/teknium']

ds = ds.groupby("dataset", group_keys=False).apply(
    lambda x: x.sample(n=min(20, len(x)), replace=False)
)

ds.to_csv('llm_judge.csv', index=False)
ds.shape

/tmp/ipykernel_17730/1410049540.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ds = ds.groupby("dataset", group_keys=False).apply(


(100, 8)